In [1]:
import gymnasium as gym          # falls back to 'import gym' if needed
import numpy as np
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical


In [2]:
# ---------- Hyper‑parameters ----------
ENV_ID            = "CartPole-v1"
TOTAL_TIMESTEPS   = 200000
UPDATE_EVERY      = 128          # how many env steps to collect before an update
EPOCHS            = 10            # gradient epochs per update
MINIBATCH_SIZE    = 32
GAMMA             = 0.99
GAE_LAMBDA        = 0.95          # λ for Generalised Advantage Estimation
CLIP_EPS          = 0.2
POLICY_LR         = 2.5e-4
VALUE_LR          = 1e-3
ENTROPY_COEF      = 0.01
VALUE_COEF        = 0.5
MAX_GRAD_NORM     = 0.5
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# -------------------------------------


In [3]:
class ActorCritics(nn.Module):
    def __init__(self,obs_dim,actions_dim):
        super(ActorCritics,self).__init__()

        self.shared = nn.Sequential(
            nn.Linear(obs_dim,64), nn.Tanh(),
            nn.Linear(64,64), nn.Tanh
        )
        self.actor = nn.Linear(64,actions_dim)
        self.critics = nn.Linear(64,1)

    def forward(self,x):
        x = self.shared(x)

        return self.actor(x), self.critics(x)

In [4]:
def compute_gae(rewards, values, dones, next_value):

    adv = np.zeros_like(rewards)
    gae = 0

    for t in reversed(range(len(rewards))):
        
        mask = 1 - dones[t]
        delta = rewards[t] + GAMMA * next_value * mask - value[t] 
        gae = delta + GAMMA * GAE_LAMBDA * gae
        
        adv[t] = gae

        next_value = values[t]

    returns = adv + values
    return adv, returns

In [5]:
def make_env(env_id):
    return env.make(env_id)

In [16]:
# ──────────────────────────────────────
def train():
    env   = make_env(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    model = ActorCritic(obs_dim, act_dim).to(DEVICE)

    optim_policy = torch.optim.Adam(list(model.shared.parameters()) + list(model.actor.parameters()), lr = POLICY_LR)
    optim_value = torch.optim.Adam(model.critics.parameters(), lr = VALUE_LR)

    # Storage for one rollout


    # Storage for one rollout
    obs_buf    = []
    act_buf    = []
    logp_buf   = []
    rew_buf    = []
    val_buf    = []
    done_buf   = []

    obs, _ = env.reset(seed=0)
    global_step = 0
    episode_returns = []
    episode_return = 0 

    while global_step < TOTAL_TIMESTEPS:
        # ─ Collect one rollout ─
        for _ in range(UPDATE_EVERY):
            global_step += 1
            obs_tensor = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
            logits, value = model(obs_tensor)
            dist = Categorical(logits=logits)
            action = dist.sample()
            logp   = dist.log_prob(action.item())

            next_obs, reward, terminate, truncate, _ = env.step(action.item())
            episode_return += reward
            
            done = terminate or truncate
            
            # Store transition
            obs_buf.append(obs_tensor)
            act_buf.append(action)
            logp_buf.append(logp)
            rew_buf.append(torch.tensor(reward, dtype=torch.float32, device=DEVICE))
            val_buf.append(value.squeeze())
            done_buf.append(torch.tensor(done, dtype=torch.float32, device=DEVICE))
            obs = next_obs
            if done:
                episode_returns.append(episode_return)
                episode_return = 0
                obs, _ = env.reset()


        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32).to(DEVICE)
            _, next_value = model(obs_t)
            next_value = next_value.squeeze()
                
        obs_t  = torch.stack(obs_buf)
        act_t  = torch.stack(act_buf)
        logp_t = torch.stack(logp_buf)
        rew_t  = torch.stack(rew_buf)
        val_t  = torch.stack(val_buf)
        done_t = torch.stack(done_buf)

        adv, returns = compute_gae(rew_buf, val_buf, dones, next_value)
        adv = (adv - adv.mean())/(adv.std() + 1e-8)    # normalize adv
        
        obs_buf.clear(); act_buf.clear(); logp_buf.clear()
        rew_buf.clear(); val_buf.clear(); done_buf.clear()



        #now training based on what we have so far

        # we do training couple of time with the same data

        batch_size = len(obs_t)
        inds = np.arange(batch_size)
        
        for _ in range(EPOCHS):
            np.random.shuffle(inds)
            for start in range(0, batch_size, MINIBATCH_SIZE):

                mb_inds = inds[start:start+MINIBATCH_SIZE]

                mb_obs   = obs_t[mb_inds]
                mb_act   = act_t[mb_inds]
                mb_logp  = logp_t[mb_inds]
                mb_adv   = adv[mb_inds]
                mb_ret   = ret[mb_inds]


                #forward
                logits,value = model(mb_obs)
                dist = Categorical(logits)
                new_logp = dist.log_prob(mb_act)
                #entropy = dist.entropy().mean() # block user from fast confidence

                ratio = ( new_logp - mb_logp).exp()

                unclipped = ratio * mb_adv
                clipped   = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * mb_adv
                policy_loss = -torch.min(unclipped, clipped).mean() # min for Take the worse case to be conservative, less aggressive


                # Value loss (MSE)
                value_loss = (mb_ret - value.squeeze()).pow(2).mean()


                                # Back‑prop
                optim_policy.zero_grad()
                optim_value.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optim_policy.step()
                optim_value.step()

                        # ─ Logging ─
        if len(episode_returns):
            print(f"Steps: {global_step:7d} | "
                  f"Mean episode return: {np.mean(episode_returns[-10:]):6.2f}")

    env.close()
    print("Training finished!")

                

In [13]:
if __name__ == "__main__":
    train()
